# Modelos econométricos (M0–M6) – Panel S&P 500 (2015–2025)

Este notebook estima las especificaciones M0–M6 con `PanelOLS` sobre el panel mensual ya construido.

**Input:**
- `data/clean/panel_master.parquet`

**Output:**
- Tablas: `outputs/tables/table_m0.tex` … `table_m6.tex` (+ backup `.csv`)
- Logs: `outputs/logs/`

**Notas:**
- Este notebook NO reconstruye spreads, betas, ivol, shocks de mercado/crédito ni variables contables.
- La dependiente se construye acá para replicar exactamente el notebook original:
  - filtra `spread_mean_bps > 0`
  - `log_spread_mean_bps = log(spread_mean_bps)`

## 1) Imports

In [ ]:
# ============================================================
# 1) Imports
# ============================================================

from __future__ import annotations

from pathlib import Path
from datetime import datetime
import logging
import warnings
import sys

import numpy as np
import pandas as pd

from linearmodels.panel import PanelOLS


In [ ]:
# ============================================================
# 2) Config + Paths
# ============================================================

ROOT = Path.cwd()

PANEL_PATH  = ROOT / "data" / "clean" / "panel_master.parquet"

OUTPUTS_DIR = ROOT / "outputs"
TABLES_DIR  = OUTPUTS_DIR / "tables"
LOGS_DIR    = OUTPUTS_DIR / "logs"

EXPORT_TEX = True
EXPORT_CSV = True

# Mantener este valor igual al notebook original
WINSOR_P = 0.01


def _ensure_dirs() -> None:
    TABLES_DIR.mkdir(parents=True, exist_ok=True)
    LOGS_DIR.mkdir(parents=True, exist_ok=True)

### 3) Logging + reproducibilidad

In [ ]:

# ============================================================
# 3) Logging + reproducibilidad
# ============================================================

def _setup_logger() -> logging.Logger:
    _ensure_dirs()

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = LOGS_DIR / f"models_run_{ts}.log"

    logger = logging.getLogger("models")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

    # Console
    ch = logging.StreamHandler(sys.stdout)
    ch.setFormatter(fmt)
    logger.addHandler(ch)

    # File
    fh = logging.FileHandler(log_path, encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    logger.info("Log file: %s", log_path.as_posix())
    return logger


logger = _setup_logger()

warnings.filterwarnings("default")  # no escondemos warnings por default

logger.info("Python: %s", sys.version.split()[0])
logger.info("pandas: %s | numpy: %s", pd.__version__, np.__version__)
try:
    import linearmodels
    logger.info("linearmodels: %s", linearmodels.__version__)
except Exception:
    logger.info("linearmodels: version not available")

logger.info("PANEL_PATH exists: %s | %s", PANEL_PATH.exists(), PANEL_PATH.as_posix())

## 4) Carga del panel limpio

**Input:**
- `data/clean/panel_master.parquet`

**Output:**
- `df` (DataFrame en memoria, todavía sin construir la dependiente)

**Chequeos:**
- Existe el archivo de input.
- `issuer` y `date` presentes.
- Unicidad de llave `issuer × date`.
- Orden temporal y rango básico de fechas.
- Presencia de columnas core que usan los modelos (sin reconstruir nada).

**Notas:**
- No se leen archivos de `data/raw/`.
- No se recalculan variables (spreads, beta, ivol, shocks, fundamentals).

In [ ]:
# ============================================================
# 4) Load panel (único punto de entrada)
# ============================================================

# --- Chequeo de input ---
if not PANEL_PATH.exists():
    raise FileNotFoundError(
        f"No existe el panel limpio en {PANEL_PATH.as_posix()}. "
        "Corré primero el Notebook 2 (Construcción_Panel.ipynb)."
    )

logger.info("Leyendo panel: %s", PANEL_PATH.as_posix())
df = pd.read_parquet(PANEL_PATH)
logger.info("Panel cargado | shape=%s", df.shape)

# --- Normalización mínima de tipos ---
# (sin cambiar lógica económica: solo tipos / orden)
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

# --- Chequeos de columnas mínimas ---
required_min_cols = ["issuer", "date", "spread_mean_bps"]
missing = [c for c in required_min_cols if c not in df.columns]
if missing:
    raise KeyError(f"Faltan columnas mínimas en panel_master: {missing}")

# --- Chequeo llave issuer×date ---
key_dups = df.duplicated(subset=["issuer", "date"]).sum()
if key_dups > 0:
    # No lo arreglo silenciosamente: esto es un bug aguas arriba.
    raise ValueError(f"Hay {key_dups} duplicados en la llave (issuer, date).")

# --- Rango temporal básico ---
min_date = df["date"].min()
max_date = df["date"].max()
logger.info("Rango fechas | min=%s | max=%s", str(min_date)[:10], str(max_date)[:10])

# --- Presencia de columnas core para modelos (chequeo “soft”) ---
# Nota: no forzamos todas acá porque algunas se usan solo en ciertos modelos.
core_candidates = [
    # shocks agregados
    "mkt_ret", "mkt_vol_60d", "credit_level", "credit_slope",
    # firm exposure (crédito)
    "crc_level_beta", "crc_slope_beta",
    # riesgo idiosincrático y sector
    "ivol", "ivol_sector",
    # market power
    "market_share", "market_share_w",
    # controles típicos
    "leverage", "log_assets", "cash_to_assets", "current_ratio",
]
present = [c for c in core_candidates if c in df.columns]
missing_soft = [c for c in core_candidates if c not in df.columns]

logger.info("Columnas core presentes (%d/%d): %s", len(present), len(core_candidates), present)
if missing_soft:
    logger.warning("Columnas core ausentes (puede ser OK si no se usan en M0–M6): %s", missing_soft)

# --- Orden consistente ---
df = df.sort_values(["issuer", "date"]).reset_index(drop=True)

# --- Snapshot rápido de muestra ---
n_issuers = df["issuer"].nunique()
n_months = df["date"].nunique()
logger.info("Cobertura | issuers=%d | months=%d", n_issuers, n_months)

df.head(3)

## 5) Variable dependiente (replicación exacta)

**Input:**
- `df` cargado desde `panel_master`

**Output:**
- `df` filtrado con `spread_mean_bps > 0`
- `log_spread_mean_bps = log(spread_mean_bps)`

**Chequeos:**
- Conteo de observaciones antes/después del filtro.
- Verificación de no-inf / no-nan en la dependiente.

**Notas:**
- Este filtro y transformación se aplican UNA sola vez al inicio para replicar el notebook original.

In [ ]:
# ============================================================
# 5) Dependiente (replicación exacta del notebook original)
# ============================================================

n0 = len(df)

# 1) Filtro exacto: spread_mean_bps > 0
df = df[df["spread_mean_bps"] > 0].copy()

n1 = len(df)
dropped = n0 - n1
logger.info(
    "Filtro dependiente: spread_mean_bps > 0 | before=%d | after=%d | dropped=%d (%.2f%%)",
    n0, n1, dropped, (dropped / n0 * 100.0) if n0 > 0 else 0.0
)

# 2) Log exacto
df["log_spread_mean_bps"] = np.log(df["spread_mean_bps"])

# --- Chequeos simples ---
bad_log = (~np.isfinite(df["log_spread_mean_bps"])).sum()
if bad_log > 0:
    # No debería pasar por el filtro >0, pero no lo dejamos silencioso.
    raise ValueError(f"Hay {bad_log} valores no finitos en log_spread_mean_bps.")

logger.info("Dependiente construida | log_spread_mean_bps OK | n=%d", len(df))

df[["issuer", "date", "spread_mean_bps", "log_spread_mean_bps"]].head(3)

## 6) Helpers (winsorización y export)

**Input:**
- `df` (panel ya filtrado y con dependiente)
- Config: `WINSOR_P`, `TABLES_DIR`, flags de export

**Output:**
- Funciones reutilizables:
  - `winsorize_series()`
  - `apply_winsor()`
  - `export_table()`

**Chequeos:**
- La winsorización no cambia el tamaño del panel (solo recorta extremos).
- `export_table()` crea archivos en `outputs/tables/`.

**Notas:**
- La winsorización se aplica **por modelo** (según lista `winsor_cols` de cada spec).
- No se winsoriza nada “por default”.

In [ ]:
# ============================================================
# 6) Helpers
# ============================================================

from typing import Iterable, Optional, Sequence, Tuple


def winsorize_series(s: pd.Series, p: float) -> pd.Series:
    """
    Winsoriza una serie a nivel p y 1-p.
    Importante: mantener p idéntico al notebook original.
    """
    s_num = pd.to_numeric(s, errors="coerce")
    lo, hi = s_num.quantile([p, 1 - p])
    return s_num.clip(lo, hi)


def apply_winsor(df_in: pd.DataFrame, cols: Sequence[str], p: float) -> pd.DataFrame:
    """
    Aplica winsorización a columnas específicas.
    - No modifica df_in in-place.
    - Si una columna no existe, levanta error (para evitar drift silencioso).
    """
    if not cols:
        return df_in

    df_out = df_in.copy()
    missing = [c for c in cols if c not in df_out.columns]
    if missing:
        raise KeyError(f"Winsor: faltan columnas requeridas: {missing}")

    for c in cols:
        df_out[c] = winsorize_series(df_out[c], p=p)

    return df_out


def export_table(
    table_df: pd.DataFrame,
    name: str,
    index: bool = False,
    float_format: Optional[str] = None,
) -> Tuple[Optional[Path], Optional[Path]]:
    """
    Exporta una tabla a outputs/tables como .csv y/o .tex.
    Devuelve (csv_path, tex_path).
    """
    _ensure_dirs()

    csv_path: Optional[Path] = None
    tex_path: Optional[Path] = None

    if EXPORT_CSV:
        csv_path = TABLES_DIR / f"{name}.csv"
        table_df.to_csv(csv_path, index=index)
        logger.info("Export CSV: %s", csv_path.as_posix())

    if EXPORT_TEX:
        tex_path = TABLES_DIR / f"{name}.tex"
        table_df.to_latex(tex_path, index=index, float_format=float_format)
        logger.info("Export TeX: %s", tex_path.as_posix())

    return csv_path, tex_path

## 7) Preparación del panel para PanelOLS

**Input:**
- `df` (ya filtrado por `spread_mean_bps > 0` y con `log_spread_mean_bps`)

**Output:**
- `df_panel` indexado como panel: `issuer × date` (MultiIndex), ordenado

**Chequeos:**
- Unicidad de la llave `issuer × date`.
- `date` sin nulos.
- Índice ordenado.

**Notas:**
- No se dropean NA globalmente.
- Los NA se manejan **por modelo** (se dropean solo en columnas usadas por esa spec).

In [ ]:
# ============================================================
# 7) Panel index + NA policy (drop por modelo)
# ============================================================

# --- Chequeos previos ---
if df["date"].isna().any():
    n_bad = df["date"].isna().sum()
    raise ValueError(f"Hay {n_bad} fechas nulas en `date`. Esto debería resolverse en el Notebook 2.")

# Re-chequeo de unicidad (ya lo hicimos en carga, pero acá aseguramos que no se rompió)
dups = df.duplicated(subset=["issuer", "date"]).sum()
if dups > 0:
    raise ValueError(f"Hay {dups} duplicados en (issuer, date) antes de set_index.")

# --- Set MultiIndex ---
df_panel = df.set_index(["issuer", "date"]).sort_index()

logger.info(
    "Panel indexado | nobs=%d | issuers=%d | months=%d",
    df_panel.shape[0],
    df_panel.index.get_level_values(0).nunique(),
    df_panel.index.get_level_values(1).nunique(),
)

# --- Helper NA policy: dropear solo columnas requeridas por cada modelo ---
def drop_na_for_cols(df_in: pd.DataFrame, cols: Sequence[str]) -> pd.DataFrame:
    """
    Dropea NA solo en las columnas usadas por una especificación.
    No toca columnas irrelevantes para esa spec.
    """
    missing = [c for c in cols if c not in df_in.columns]
    if missing:
        raise KeyError(f"drop_na_for_cols: faltan columnas requeridas: {missing}")

    before = df_in.shape[0]
    df_out = df_in.dropna(subset=list(cols))
    after = df_out.shape[0]

    # Log corto, útil para trazabilidad
    if after < before:
        logger.info("drop_na_for_cols | dropped=%d (%.2f%%) | kept=%d",
                    before - after,
                    (before - after) / before * 100.0 if before > 0 else 0.0,
                    after)
    return df_out


df_panel.head(3)

## 8) Especificaciones M0–M6

**Input:**
- `df_panel` (panel indexado `issuer × date`)
- Columnas ya existentes en `panel_master`

**Output:**
- `SPECS`: diccionario con la definición de cada modelo
  - dependiente
  - regresores (RHS)
  - efectos fijos
  - columnas a winsorizar
  - configuración de covarianza

**Notas:**
- La winsorización se aplica a la dependiente y a los regresores de cada modelo.
- Las interacciones se construyen en la sección siguiente.

In [ ]:
# ============================================================
# 8) SPECS M0–M6
# ============================================================

DEP_VAR = "log_spread_mean_bps"

BASE_FIRM_RHS = [
    "market_share_w",
    "ivol_sector",
    "residual_maturity_mean",
    "rollover_12m",
    "cash_to_assets",
    "log_assets",
    "leverage",
]

MACRO_RHS_NO_MKTRET = [
    "mkt_vol_60d",
    "credit_level",
    "credit_slope",
]

MACRO_RHS_WITH_MKTRET = ["mkt_ret"] + MACRO_RHS_NO_MKTRET

MKT_INTERACTIONS = [
    "mkt_ret_x_rollover",
    "mkt_ret_x_leverage",
    "mkt_ret_x_power",
]

CL_INTERACTIONS = [
    "cl_x_rollover",
    "cl_x_leverage",
    "cl_x_maturity",
    "cl_x_power",
]

ADD_SLOPE_INTERACTIONS = False

CS_INTERACTIONS = [
    "cs_x_rollover",
    "cs_x_leverage",
    "cs_x_maturity",
    "cs_x_power",
]

DK_M0_M1 = {"cov_type": "driscoll-kraay"}

DK_M2_PLUS = {
    "cov_type": "kernel",
    "kernel": "bartlett",
    "bandwidth": 4,
}

SPECS = {
    "M0": {
        "dep": DEP_VAR,
        "rhs": BASE_FIRM_RHS,
        "entity_effects": True,
        "time_effects": True,
        "winsor_cols": [DEP_VAR] + BASE_FIRM_RHS,
        "cluster_entity": True,
        "dk_kwargs": DK_M0_M1,
    },
    "M1": {
        "dep": DEP_VAR,
        "rhs": BASE_FIRM_RHS + MACRO_RHS_NO_MKTRET,
        "entity_effects": True,
        "time_effects": False,
        "winsor_cols": [DEP_VAR] + BASE_FIRM_RHS + MACRO_RHS_NO_MKTRET,
        "cluster_entity": True,
        "dk_kwargs": DK_M0_M1,
    },
    "M2": {
        "dep": DEP_VAR,
        "rhs": BASE_FIRM_RHS + MACRO_RHS_WITH_MKTRET,
        "entity_effects": True,
        "time_effects": False,
        "winsor_cols": [DEP_VAR] + BASE_FIRM_RHS + MACRO_RHS_WITH_MKTRET,
        "cluster_entity": True,
        "dk_kwargs": DK_M2_PLUS,
    },
    "M3": {
        "dep": DEP_VAR,
        "rhs": (
            BASE_FIRM_RHS
            + ["mkt_ret"]
            + MKT_INTERACTIONS
            + MACRO_RHS_NO_MKTRET
        ),
        "entity_effects": True,
        "time_effects": False,
        "winsor_cols": (
            [DEP_VAR]
            + BASE_FIRM_RHS
            + ["mkt_ret"]
            + MKT_INTERACTIONS
            + MACRO_RHS_NO_MKTRET
        ),
        "cluster_entity": True,
        "dk_kwargs": DK_M2_PLUS,
    },
    "M4": {
        "dep": DEP_VAR,
        "rhs": (
            BASE_FIRM_RHS
            + ["credit_level", "credit_slope"]
            + ["mkt_ret", "mkt_vol_60d"]
        ),
        "entity_effects": True,
        "time_effects": False,
        "winsor_cols": (
            [DEP_VAR]
            + BASE_FIRM_RHS
            + ["credit_level", "credit_slope", "mkt_ret", "mkt_vol_60d"]
        ),
        "cluster_entity": True,
        "dk_kwargs": DK_M2_PLUS,
    },
    "M5": {
        "dep": DEP_VAR,
        "rhs": (
            BASE_FIRM_RHS
            + ["credit_level", "credit_slope"]
            + CL_INTERACTIONS
            + ["mkt_ret", "mkt_vol_60d"]
            + (CS_INTERACTIONS if ADD_SLOPE_INTERACTIONS else [])
        ),
        "entity_effects": True,
        "time_effects": False,
        "winsor_cols": (
            [DEP_VAR]
            + BASE_FIRM_RHS
            + ["credit_level", "credit_slope"]
            + CL_INTERACTIONS
            + ["mkt_ret", "mkt_vol_60d"]
            + (CS_INTERACTIONS if ADD_SLOPE_INTERACTIONS else [])
        ),
        "cluster_entity": True,
        "dk_kwargs": DK_M2_PLUS,
    },
    "M6": {
        "dep": DEP_VAR,
        "rhs": (
            BASE_FIRM_RHS
            + ["mkt_ret", "mkt_vol_60d", "credit_level", "credit_slope"]
            + MKT_INTERACTIONS
            + CL_INTERACTIONS
        ),
        "entity_effects": True,
        "time_effects": False,
        "winsor_cols": (
            [DEP_VAR]
            + BASE_FIRM_RHS
            + ["mkt_ret", "mkt_vol_60d", "credit_level", "credit_slope"]
            + MKT_INTERACTIONS
            + CL_INTERACTIONS
        ),
        "cluster_entity": True,
        "dk_kwargs": DK_M2_PLUS,
    },
}


def required_cols_for_spec(spec: dict) -> list[str]:
    return sorted(set([spec["dep"]] + list(spec["rhs"])))


for key, spec in SPECS.items():
    logger.info(
        "%s | rhs=%d | entity_FE=%s | time_FE=%s",
        key,
        len(spec["rhs"]),
        spec["entity_effects"],
        spec["time_effects"],
    )

## 9) Interacciones

**Input:**
- `df_panel` (panel indexado `issuer × date`)
- Variables base ya presentes en el panel:
  - `mkt_ret`, `credit_level`
  - `rollover_12m`, `leverage`, `residual_maturity_mean`, `market_share_w`

**Output:**
- Columnas de interacción (si hacen falta para algún modelo):
  - Mercado: `mkt_ret_x_rollover`, `mkt_ret_x_leverage`, `mkt_ret_x_power`
  - Crédito (level): `cl_x_rollover`, `cl_x_leverage`, `cl_x_maturity`, `cl_x_power`

**Chequeos:**
- No pisa columnas existentes.
- Verifica que existan las columnas base necesarias.
- Genera solo lo necesario según `SPECS`.

In [ ]:
# ============================================================
# 9) Interacciones
# ============================================================

def _need_any(models: Sequence[str], cols: Sequence[str]) -> bool:
    """True si alguna spec usa alguna de las columnas `cols`."""
    for m in models:
        used = set(SPECS[m]["rhs"])
        if any(c in used for c in cols):
            return True
    return False


def make_interactions(df_in: pd.DataFrame, models: Sequence[str] = ("M0", "M1", "M2", "M3", "M4", "M5", "M6")) -> pd.DataFrame:
    """
    Crea columnas de interacción requeridas por las especificaciones.
    No recalcula shocks: usa columnas ya presentes en el panel.
    """
    df = df_in.copy()

    # Mercado
    need_mkt = _need_any(models, MKT_INTERACTIONS)
    if need_mkt:
        base_cols = ["mkt_ret", "rollover_12m", "leverage", "market_share_w"]
        missing = [c for c in base_cols if c not in df.columns]
        if missing:
            raise KeyError(f"Faltan columnas para interacciones de mercado: {missing}")

        new_cols = {
            "mkt_ret_x_rollover": df["mkt_ret"] * df["rollover_12m"],
            "mkt_ret_x_leverage": df["mkt_ret"] * df["leverage"],
            "mkt_ret_x_power": df["mkt_ret"] * df["market_share_w"],
        }

        for c in new_cols:
            if c in df.columns:
                raise ValueError(f"La columna ya existe y no se va a pisar: {c}")

        for c, v in new_cols.items():
            df[c] = v

        logger.info("Interacciones mercado creadas: %s", list(new_cols.keys()))

    # Crédito (level)
    need_cl = _need_any(models, CL_INTERACTIONS)
    if need_cl:
        base_cols = ["credit_level", "rollover_12m", "leverage", "residual_maturity_mean", "market_share_w"]
        missing = [c for c in base_cols if c not in df.columns]
        if missing:
            raise KeyError(f"Faltan columnas para interacciones de crédito (level): {missing}")

        new_cols = {
            "cl_x_rollover": df["credit_level"] * df["rollover_12m"],
            "cl_x_leverage": df["credit_level"] * df["leverage"],
            "cl_x_maturity": df["credit_level"] * df["residual_maturity_mean"],
            "cl_x_power": df["credit_level"] * df["market_share_w"],
        }

        for c in new_cols:
            if c in df.columns:
                raise ValueError(f"La columna ya existe y no se va a pisar: {c}")

        for c, v in new_cols.items():
            df[c] = v

        logger.info("Interacciones crédito (level) creadas: %s", list(new_cols.keys()))

    # Crédito (slope) (desactivado por default)
    if ADD_SLOPE_INTERACTIONS and _need_any(models, CS_INTERACTIONS):
        base_cols = ["credit_slope", "rollover_12m", "leverage", "residual_maturity_mean", "market_share_w"]
        missing = [c for c in base_cols if c not in df.columns]
        if missing:
            raise KeyError(f"Faltan columnas para interacciones de crédito (slope): {missing}")

        new_cols = {
            "cs_x_rollover": df["credit_slope"] * df["rollover_12m"],
            "cs_x_leverage": df["credit_slope"] * df["leverage"],
            "cs_x_maturity": df["credit_slope"] * df["residual_maturity_mean"],
            "cs_x_power": df["credit_slope"] * df["market_share_w"],
        }

        for c in new_cols:
            if c in df.columns:
                raise ValueError(f"La columna ya existe y no se va a pisar: {c}")

        for c, v in new_cols.items():
            df[c] = v

        logger.info("Interacciones crédito (slope) creadas: %s", list(new_cols.keys()))

    return df


df_panel = make_interactions(df_panel)

# Snapshot rápido
cols_show = [c for c in (MKT_INTERACTIONS + CL_INTERACTIONS) if c in df_panel.columns]
df_panel[cols_show].head(3)

## 10) Estimación (runner)

**Input:**
- `df_panel` (MultiIndex `issuer × date`)
- `SPECS` (definiciones M0–M6)
- Helpers: `drop_na_for_cols()`, `apply_winsor()`

**Output:**
- `results`: dict con resultados `PanelOLS` por modelo
- `meta`: resumen por modelo (nobs, n_firms, n_periods)

**Chequeos:**
- Se dropean NA solo en columnas requeridas por cada modelo.
- La winsorización se aplica solo a las columnas listadas en `winsor_cols`.
- Cluster por entidad (`issuer`) en todos los modelos.

In [ ]:
# ============================================================
# 10) Estimación (runner)
# ============================================================

def fit_panelols(
    df_in: pd.DataFrame,
    dep: str,
    rhs: Sequence[str],
    entity_effects: bool,
    time_effects: bool,
    cluster_entity: bool = True,
):
    """
    Ajusta PanelOLS con clustering por entidad.
    """
    y = df_in[dep]
    X = df_in[list(rhs)]

    model = PanelOLS(y, X, entity_effects=entity_effects, time_effects=time_effects)

    fit_res = model.fit(
        cov_type="clustered",
        cluster_entity=cluster_entity,
    )
    return fit_res


def fit_dk(
    df_in: pd.DataFrame,
    dep: str,
    rhs: Sequence[str],
    entity_effects: bool,
    time_effects: bool,
    dk_kwargs: dict,
):
    """
    Ajusta PanelOLS con cov_type DK/kernel según `dk_kwargs`.
    """
    y = df_in[dep]
    X = df_in[list(rhs)]

    model = PanelOLS(y, X, entity_effects=entity_effects, time_effects=time_effects)
    fit_res = model.fit(**dk_kwargs)
    return fit_res


def run_models(
    df_panel_in: pd.DataFrame,
    specs: dict,
    winsor_p: float,
) -> tuple[dict, pd.DataFrame]:
    """
    Corre M0–M6:
    - construye dataset por modelo (drop NA por cols requeridas)
    - aplica winsor por modelo
    - estima clustered y DK (según spec)
    """
    results: dict = {}
    meta_rows = []

    for key, spec in specs.items():
        dep = spec["dep"]
        rhs = spec["rhs"]

        req_cols = required_cols_for_spec(spec)

        logger.info("=== %s ===", key)
        logger.info("dep=%s | rhs=%d", dep, len(rhs))

        # Drop NA solo en columnas necesarias
        df_m = drop_na_for_cols(df_panel_in, req_cols)

        # Winsor por modelo
        df_m = apply_winsor(df_m, spec["winsor_cols"], p=winsor_p)

        nobs = df_m.shape[0]
        nfirms = df_m.index.get_level_values(0).nunique()
        nperiods = df_m.index.get_level_values(1).nunique()

        logger.info("Muestra %s | nobs=%d | firms=%d | periods=%d", key, nobs, nfirms, nperiods)

        # Estimación clustered (principal)
        res_cluster = fit_panelols(
            df_in=df_m,
            dep=dep,
            rhs=rhs,
            entity_effects=spec["entity_effects"],
            time_effects=spec["time_effects"],
            cluster_entity=spec.get("cluster_entity", True),
        )

        # Estimación DK/kernel (robustez)
        res_dk = fit_dk(
            df_in=df_m,
            dep=dep,
            rhs=rhs,
            entity_effects=spec["entity_effects"],
            time_effects=spec["time_effects"],
            dk_kwargs=spec["dk_kwargs"],
        )

        results[key] = {
            "clustered": res_cluster,
            "dk": res_dk,
            "nobs": nobs,
            "nfirms": nfirms,
            "nperiods": nperiods,
        }

        meta_rows.append(
            {
                "model": key,
                "nobs": nobs,
                "n_firms": nfirms,
                "n_periods": nperiods,
                "r2_within": getattr(res_cluster, "rsquared_within", np.nan),
                "r2_overall": getattr(res_cluster, "rsquared_overall", np.nan),
            }
        )

        logger.info("%s | clustered r2_within=%.4f", key, meta_rows[-1]["r2_within"])

    meta = pd.DataFrame(meta_rows).set_index("model")
    return results, meta


results, meta = run_models(df_panel, SPECS, winsor_p=WINSOR_P)

meta

## 11) Tablas y export

**Input:**
- `results` (salida de `run_models`)
- `meta` (resumen por modelo)

**Output:**
- `outputs/tables/table_m0.tex` … `outputs/tables/table_m6.tex`
- Backups: `outputs/tables/table_m0.csv` … `table_m6.csv`
- Tabla resumen: `outputs/tables/table_meta.tex/.csv`

**Notas:**
- Se exporta una tabla por modelo con coeficientes y errores estándar.
- Por defecto se exporta el ajuste con covarianza clustered.
- La versión DK se exporta como tabla separada.

In [ ]:
# ============================================================
# 11) Tablas + export
# ============================================================

def _format_pstars(p: float) -> str:
    if pd.isna(p):
        return ""
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""


def result_to_table(res) -> pd.DataFrame:
    """
    Convierte un resultado de PanelOLS a tabla de coeficientes.
    """
    params = res.params
    ses = res.std_errors
    tstats = res.tstats
    pvals = res.pvalues

    out = pd.DataFrame(
        {
            "coef": params,
            "std_err": ses,
            "tstat": tstats,
            "pval": pvals,
        }
    )
    out["stars"] = out["pval"].apply(_format_pstars)
    return out


def export_model_tables(results_dict: dict) -> None:
    """
    Exporta:
    - table_mX.*  (clustered)
    - table_mX_dk.* (DK/kernel)
    """
    # Meta
    export_table(meta.reset_index(), name="table_meta", index=False)

    for key in ["M0", "M1", "M2", "M3", "M4", "M5", "M6"]:
        res_cluster = results_dict[key]["clustered"]
        res_dk = results_dict[key]["dk"]

        tab_cluster = result_to_table(res_cluster).reset_index().rename(columns={"index": "variable"})
        tab_dk = result_to_table(res_dk).reset_index().rename(columns={"index": "variable"})

        export_table(tab_cluster, name=f"table_{key.lower()}", index=False)
        export_table(tab_dk, name=f"table_{key.lower()}_dk", index=False)

        logger.info("Tablas exportadas: %s", key)


export_model_tables(results)

## 12) QA final

**Input:**
- `meta`
- `outputs/tables/` y `outputs/logs/`

**Output:**
- Checks de existencia de archivos esperados.
- Resumen final por modelo (nobs, firms, r2).

**Chequeos:**
- Están generadas las tablas `table_m0` … `table_m6` (tex + csv).
- Están generadas las tablas DK correspondientes.
- El panel quedó indexado correctamente.

In [ ]:
# ============================================================
# 12) QA final
# ============================================================

expected_tables = []
for k in ["m0", "m1", "m2", "m3", "m4", "m5", "m6"]:
    expected_tables += [
        TABLES_DIR / f"table_{k}.csv",
        TABLES_DIR / f"table_{k}.tex",
        TABLES_DIR / f"table_{k}_dk.csv",
        TABLES_DIR / f"table_{k}_dk.tex",
    ]

expected_tables += [
    TABLES_DIR / "table_meta.csv",
    TABLES_DIR / "table_meta.tex",
]

missing_files = [p for p in expected_tables if not p.exists()]
if missing_files:
    logger.error("Faltan outputs (%d):", len(missing_files))
    for p in missing_files:
        logger.error(" - %s", p.as_posix())
    raise FileNotFoundError("QA falló: faltan archivos de salida esperados.")
else:
    logger.info("QA outputs: OK | %d archivos encontrados", len(expected_tables))

# Resumen final de corrida
logger.info("Resumen por modelo (clustered):")
display_cols = ["nobs", "n_firms", "n_periods", "r2_within", "r2_overall"]
meta_show = meta[display_cols].copy()

# Logueo compacto
for model, row in meta_show.iterrows():
    logger.info(
        "%s | nobs=%d | firms=%d | periods=%d | r2_within=%.4f",
        model,
        int(row["nobs"]),
        int(row["n_firms"]),
        int(row["n_periods"]),
        float(row["r2_within"]) if pd.notna(row["r2_within"]) else np.nan,
    )

meta_show

## 13) Outputs generados

La corrida de este notebook genera los siguientes archivos:

### Tablas principales (clustered)
- `outputs/tables/table_m0.tex`
- `outputs/tables/table_m1.tex`
- `outputs/tables/table_m2.tex`
- `outputs/tables/table_m3.tex`
- `outputs/tables/table_m4.tex`
- `outputs/tables/table_m5.tex`
- `outputs/tables/table_m6.tex`

Backups en CSV:
- `outputs/tables/table_m0.csv`
- …
- `outputs/tables/table_m6.csv`

### Tablas robustez (DK / kernel)
- `outputs/tables/table_m0_dk.tex`
- …
- `outputs/tables/table_m6_dk.tex`

Backups en CSV:
- `outputs/tables/table_m0_dk.csv`
- …
- `outputs/tables/table_m6_dk.csv`

### Tabla resumen de muestra
- `outputs/tables/table_meta.tex`
- `outputs/tables/table_meta.csv`

### Logs
- `outputs/logs/models_run_YYYYMMDD_HHMM.log`

---

**Replicación:**
1. Ejecutar el Notebook 2 (Construcción_Panel.ipynb).
2. Ejecutar este notebook completo (Run All).
3. Verificar que la sección 12 (QA) no arroje errores.